# Replay verification, no model

Applies 8 patches (that resolved LOCALLY) to the base snapshots, applies the official
test_patch, and runs exactly the test nodes swegemma would run. CPU only.

Purpose: is the local sandbox the reason these tasks resolve locally but not on Kaggle?


In [ ]:

import base64, json, os, subprocess, sys
from pathlib import Path

print('input:', sorted(os.listdir('/kaggle/input')))
TASKS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'tasks.jsonl' in files:
        TASKS = Path(root) / 'tasks.jsonl'
        break
SNAPS = None
for root, dirs, files in os.walk('/kaggle/input'):
    if any(f.endswith('.tgz') for f in files):
        SNAPS = Path(root)
        break
print('tasks:', TASKS)
print('snaps:', SNAPS)
assert TASKS and SNAPS

PATCHES = json.loads("""{"fastapi_14851": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvYXBwbGljYXRpb25zLnB5IGIvZmFzdGFwaS9hcHBsaWNhdGlvbnMucHkKaW5kZXggMzQwY2FiZmMyLi41NjMwN2YwNTggMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvYXBwbGljYXRpb25zLnB5CisrKyBiL2Zhc3RhcGkvYXBwbGljYXRpb25zLnB5CkBAIC00NTY3LDYgKzQ1NjcsNyBAQCBjbGFzcyBGYXN0QVBJKFN0YXJsZXR0ZSk6CiAgICAgICAgICAgICAgICAgIiIiCiAgICAgICAgICAgICApLAogICAgICAgICBdLAorICAgICAgICBsaWZlc3BhbjogT3B0aW9uYWxbTGlmZXNwYW5dID0gTm9uZSwKICAgICApIC0+IENhbGxhYmxlW1tEZWNvcmF0ZWRDYWxsYWJsZV0sIERlY29yYXRlZENhbGxhYmxlXToKICAgICAgICAgIiIiCiAgICAgICAgIEFkZCBhbiBldmVudCBoYW5kbGVyIGZvciB0aGUgYXBwbGljYXRpb24uCg==", "fastapi_14786": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvc2VjdXJpdHkvdXRpbHMucHkgYi9mYXN0YXBpL3NlY3VyaXR5L3V0aWxzLnB5CmluZGV4IDAwMmU2OGI0NC4uZmQzNDlhZWM3IDEwMDY0NAotLS0gYS9mYXN0YXBpL3NlY3VyaXR5L3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvc2VjdXJpdHkvdXRpbHMucHkKQEAgLTcsNCArNyw0IEBAIGRlZiBnZXRfYXV0aG9yaXphdGlvbl9zY2hlbWVfcGFyYW0oCiAgICAgaWYgbm90IGF1dGhvcml6YXRpb25faGVhZGVyX3ZhbHVlOgogICAgICAgICByZXR1cm4gIiIsICIiCiAgICAgc2NoZW1lLCBfLCBwYXJhbSA9IGF1dGhvcml6YXRpb25faGVhZGVyX3ZhbHVlLnBhcnRpdGlvbigiICIpCi0gICAgcmV0dXJuIHNjaGVtZSwgcGFyYW0KKyAgICByZXR1cm4gc2NoZW1lLCBwYXJhbS5zdHJpcCgpCg==", "fastapi_14463": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weSBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQppbmRleCA3NWZmMjYxMDIuLmNlYzFiMmQzOSAxMDA2NDQKLS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtMSw0ICsxLDUgQEAKIGltcG9ydCBodHRwLmNsaWVudAoraW1wb3J0IGNvcHkKIGltcG9ydCBpbnNwZWN0CiBpbXBvcnQgd2FybmluZ3MKIGZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBTZXF1ZW5jZQpAQCAtMzc3LDcgKzM3OCw3IEBAIGRlZiBnZXRfb3BlbmFwaV9wYXRoKAogICAgICAgICAgICAgICAgICAgICBhZGRpdGlvbmFsX3N0YXR1c19jb2RlLAogICAgICAgICAgICAgICAgICAgICBhZGRpdGlvbmFsX3Jlc3BvbnNlLAogICAgICAgICAgICAgICAgICkgaW4gcm91dGUucmVzcG9uc2VzLml0ZW1zKCk6Ci0gICAgICAgICAgICAgICAgICAgIHByb2Nlc3NfcmVzcG9uc2UgPSBhZGRpdGlvbmFsX3Jlc3BvbnNlLmNvcHkoKQorICAgICAgICAgICAgICAgICAgICBwcm9jZXNzX3Jlc3BvbnNlID0gY29weS5kZWVwY29weShhZGRpdGlvbmFsX3Jlc3BvbnNlKQogICAgICAgICAgICAgICAgICAgICBwcm9jZXNzX3Jlc3BvbnNlLnBvcCgibW9kZWwiLCBOb25lKQogICAgICAgICAgICAgICAgICAgICBzdGF0dXNfY29kZV9rZXkgPSBzdHIoYWRkaXRpb25hbF9zdGF0dXNfY29kZSkudXBwZXIoKQogICAgICAgICAgICAgICAgICAgICBpZiBzdGF0dXNfY29kZV9rZXkgPT0gIkRFRkFVTFQiOgo=", "fastapi_14301": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5IGIvZmFzdGFwaS9kZXBlbmRlbmNpZXMvdXRpbHMucHkKaW5kZXggYzVjNmI2OWJiLi5jMDkwNTc5NTMgMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvZGVwZW5kZW5jaWVzL3V0aWxzLnB5CkBAIC0xMjgsOCArMTI4LDkgQEAgZGVmIGdldF9wYXJhbWV0ZXJsZXNzX3N1Yl9kZXBlbmRhbnQoKiwgZGVwZW5kczogcGFyYW1zLkRlcGVuZHMsIHBhdGg6IHN0cikgLT4gRGUKICAgICB1c2Vfc2VjdXJpdHlfc2NvcGVzOiBMaXN0W3N0cl0gPSBbXQogICAgIGlmIGlzaW5zdGFuY2UoZGVwZW5kcywgcGFyYW1zLlNlY3VyaXR5KSBhbmQgZGVwZW5kcy5zY29wZXM6CiAgICAgICAgIHVzZV9zZWN1cml0eV9zY29wZXMuZXh0ZW5kKGRlcGVuZHMuc2NvcGVzKQorICAgIHNjb3BlID0gZ2V0YXR0cihkZXBlbmRzLCAic2NvcGUiLCBOb25lKQogICAgIHJldHVybiBnZXRfZGVwZW5kYW50KAotICAgICAgICBwYXRoPXBhdGgsIGNhbGw9ZGVwZW5kcy5kZXBlbmRlbmN5LCBzZWN1cml0eV9zY29wZXM9dXNlX3NlY3VyaXR5X3Njb3BlcworICAgICAgICBwYXRoPXBhdGgsIGNhbGw9ZGVwZW5kcy5kZXBlbmRlbmN5LCBzZWN1cml0eV9zY29wZXM9dXNlX3NlY3VyaXR5X3Njb3Blcywgc2NvcGU9c2NvcGUKICAgICApCiAKIAo=", "fastapi_14077": "ZGlmZiAtLWdpdCBhL2Zhc3RhcGkvZXhjZXB0aW9uX2hhbmRsZXJzLnB5IGIvZmFzdGFwaS9leGNlcHRpb25faGFuZGxlcnMucHkKaW5kZXggNmMyYmE3ZmVkLi43MzljYjU4MDkgMTAwNjQ0Ci0tLSBhL2Zhc3RhcGkvZXhjZXB0aW9uX2hhbmRsZXJzLnB5CisrKyBiL2Zhc3RhcGkvZXhjZXB0aW9uX2hhbmRsZXJzLnB5CkBAIC0yMSw3ICsyMSw3IEBAIGFzeW5jIGRlZiByZXF1ZXN0X3ZhbGlkYXRpb25fZXhjZXB0aW9uX2hhbmRsZXIoCiAgICAgcmVxdWVzdDogUmVxdWVzdCwgZXhjOiBSZXF1ZXN0VmFsaWRhdGlvbkVycm9yCiApIC0+IEpTT05SZXNwb25zZToKICAgICByZXR1cm4gSlNPTlJlc3BvbnNlKAotICAgICAgICBzdGF0dXNfY29kZT1IVFRQXzQyMl9VTlBST0NFU1NBQkxFX0VOVElUWSwKKyAgICAgICAgc3RhdHVzX2NvZGU9NDIyLAogICAgICAgICBjb250ZW50PXsiZGV0YWlsIjoganNvbmFibGVfZW5jb2RlcihleGMuZXJyb3JzKCkpfSwKICAgICApCiAKZGlmZiAtLWdpdCBhL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weSBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQppbmRleCA4MDg2NDZjYzIuLjQ5NDZkZTk4MSAxMDA2NDQKLS0tIGEvZmFzdGFwaS9vcGVuYXBpL3V0aWxzLnB5CisrKyBiL2Zhc3RhcGkvb3BlbmFwaS91dGlscy5weQpAQCAtNDE2LDcgKzQxNiw3IEBAIGRlZiBnZXRfb3BlbmFwaV9wYXRoKAogICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgIGRlZXBfZGljdF91cGRhdGUob3BlbmFwaV9yZXNwb25zZSwgcHJvY2Vzc19yZXNwb25zZSkKICAgICAgICAgICAgICAgICAgICAgb3BlbmFwaV9yZXNwb25zZVsiZGVzY3JpcHRpb24iXSA9IGRlc2NyaXB0aW9uCi0gICAgICAgICAgICBodHRwNDIyID0gc3RyKEhUVFBfNDIyX1VOUFJPQ0VTU0FCTEVfRU5USVRZKQorICAgICAgICAgICAgaHR0cDQyMiA9IHN0cig0MjIpCiAgICAgICAgICAgICBhbGxfcm91dGVfcGFyYW1zID0gZ2V0X2ZsYXRfcGFyYW1zKHJvdXRlLmRlcGVuZGFudCkKICAgICAgICAgICAgIGlmIChhbGxfcm91dGVfcGFyYW1zIG9yIHJvdXRlLmJvZHlfZmllbGQpIGFuZCBub3QgYW55KAogICAgICAgICAgICAgICAgIHN0YXR1cyBpbiBvcGVyYXRpb25bInJlc3BvbnNlcyJdCg==", "rich_3894": "ZGlmZiAtLWdpdCBhL3JpY2gvX2luc3BlY3QucHkgYi9yaWNoL19pbnNwZWN0LnB5CmluZGV4IDI3ZDY1Y2VjLi41MmRjZGE3NSAxMDA2NDQKLS0tIGEvcmljaC9faW5zcGVjdC5weQorKysgYi9yaWNoL19pbnNwZWN0LnB5CkBAIC0xMDAsNyArMTAwLDE1IEBAIGNsYXNzIEluc3BlY3QoSnVweXRlck1peGluKToKICAgICAgICAgICAgIGNhbGxhYmxlX25hbWUuc3R5bGl6ZShmImxpbmsgZmlsZTovL3tzb3VyY2VfZmlsZW5hbWV9IikKICAgICAgICAgc2lnbmF0dXJlX3RleHQgPSBzZWxmLmhpZ2hsaWdodGVyKF9zaWduYXR1cmUpCiAKLSAgICAgICAgcXVhbG5hbWUgPSBuYW1lIG9yIGdldGF0dHIob2JqLCAiX19xdWFsbmFtZV9fIiwgbmFtZSkKKyAgICAgICAgcXVhbG5hbWUgPSBnZXRhdHRyKG9iaiwgIl9fcXVhbG5hbWVfXyIsIE5vbmUpCisgICAgICAgIGlmIGlzaW5zdGFuY2UocXVhbG5hbWUsIHN0cik6CisgICAgICAgICAgICBxdWFsbmFtZSA9IG5hbWUgb3IgcXVhbG5hbWUKKyAgICAgICAgZWxzZToKKyAgICAgICAgICAgIHF1YWxuYW1lID0gZ2V0YXR0cihvYmosICJfX25hbWVfXyIsIE5vbmUpCisgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHF1YWxuYW1lLCBzdHIpOgorICAgICAgICAgICAgICAgIHF1YWxuYW1lID0gbmFtZSBvciBxdWFsbmFtZQorICAgICAgICAgICAgZWxzZToKKyAgICAgICAgICAgICAgICBxdWFsbmFtZSA9IG5hbWUKIAogICAgICAgICAjIElmIG9iaiBpcyBhIG1vZHVsZSwgdGhlcmUgbWF5IGJlIGNsYXNzZXMgKHdoaWNoIGFyZSBjYWxsYWJsZSkgdG8gZGlzcGxheQogICAgICAgICBpZiBpbnNwZWN0LmlzY2xhc3Mob2JqKToK", "rich_3905": "ZGlmZiAtLWdpdCBhL3JpY2gvcHJvZ3Jlc3MucHkgYi9yaWNoL3Byb2dyZXNzLnB5CmluZGV4IDQyMWUxNDExLi5mNTViOTgxZSAxMDA2NDQKLS0tIGEvcmljaC9wcm9ncmVzcy5weQorKysgYi9yaWNoL3Byb2dyZXNzLnB5CkBAIC0xMTcyLDkgKzExNzIsMTAgQEAgY2xhc3MgUHJvZ3Jlc3MoSnVweXRlck1peGluKToKIAogICAgIGRlZiBzdG9wKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgICIiIlN0b3AgdGhlIHByb2dyZXNzIGRpc3BsYXkuIiIiCi0gICAgICAgIHNlbGYubGl2ZS5zdG9wKCkKLSAgICAgICAgaWYgbm90IHNlbGYuY29uc29sZS5pc19pbnRlcmFjdGl2ZSBhbmQgbm90IHNlbGYuY29uc29sZS5pc19qdXB5dGVyOgotICAgICAgICAgICAgc2VsZi5jb25zb2xlLnByaW50KCkKKyAgICAgICAgaWYgbm90IHNlbGYuZGlzYWJsZToKKyAgICAgICAgICAgIHNlbGYubGl2ZS5zdG9wKCkKKyAgICAgICAgICAgIGlmIG5vdCBzZWxmLmNvbnNvbGUuaXNfaW50ZXJhY3RpdmUgYW5kIG5vdCBzZWxmLmNvbnNvbGUuaXNfanVweXRlcjoKKyAgICAgICAgICAgICAgICBzZWxmLmNvbnNvbGUucHJpbnQoKQogCiAgICAgZGVmIF9fZW50ZXJfXyhzZWxmKSAtPiBTZWxmOgogICAgICAgICBzZWxmLnN0YXJ0KCkK", "requests_6644": "ZGlmZiAtLWdpdCBhL3NyYy9yZXF1ZXN0cy9tb2RlbHMucHkgYi9zcmMvcmVxdWVzdHMvbW9kZWxzLnB5CmluZGV4IDQ0NTU2Mzk0Li5kZTI0NDUwMyAxMDA2NDQKLS0tIGEvc3JjL3JlcXVlc3RzL21vZGVscy5weQorKysgYi9zcmMvcmVxdWVzdHMvbW9kZWxzLnB5CkBAIC05MSw3ICs5MSw5IEBAIGNsYXNzIFJlcXVlc3RFbmNvZGluZ01peGluOgogICAgICAgICBwID0gdXJsc3BsaXQoc2VsZi51cmwpCiAKICAgICAgICAgcGF0aCA9IHAucGF0aAotICAgICAgICBpZiBub3QgcGF0aDoKKyAgICAgICAgaWYgcGF0aC5zdGFydHN3aXRoKCcvJyk6CisgICAgICAgICAgICBwYXRoID0gJy8nICsgcGF0aC5sc3RyaXAoJy8nKQorICAgICAgICBlbGlmIG5vdCBwYXRoOgogICAgICAgICAgICAgcGF0aCA9ICIvIgogCiAgICAgICAgIHVybC5hcHBlbmQocGF0aCkK", "rich_3454": "ZGlmZiAtLWdpdCBhL3JpY2gvaGlnaGxpZ2h0ZXIucHkgYi9yaWNoL2hpZ2hsaWdodGVyLnB5CmluZGV4IDI3NzE0YjI1Li5lNGM0NjJlMiAxMDA2NDQKLS0tIGEvcmljaC9oaWdobGlnaHRlci5weQorKysgYi9yaWNoL2hpZ2hsaWdodGVyLnB5CkBAIC05OCw3ICs5OCw3IEBAIGNsYXNzIFJlcHJIaWdobGlnaHRlcihSZWdleEhpZ2hsaWdodGVyKToKICAgICAgICAgICAgIHIiKD9QPG51bWJlcj4oPzwhXHcpXC0/WzAtOV0rXC4/WzAtOV0qKGVbLStdP1xkKz8pP1xifDB4WzAtOWEtZkEtRl0qKSIsCiAgICAgICAgICAgICByIig/UDxwYXRoPlxCKC9bLVx3Ll8rXSspKlwvKSg/UDxmaWxlbmFtZT5bLVx3Ll8rXSopPyIsCiAgICAgICAgICAgICByIig/PCFbXFxcd10pKD9QPHN0cj5iPycnJy4qPyg/PCFcXCknJyd8Yj8nLio/KD88IVxcKSd8Yj9cIlwiXCIuKj8oPzwhXFwpXCJcIlwifGI/XCIuKj8oPzwhXFwpXCIpIiwKLSAgICAgICAgICAgIHIiKD9QPHVybD4oZmlsZXxodHRwc3xodHRwfHdzfHdzcyk6Ly9bLTAtOWEtekEtWiRfKyFgKCksLj8vOzomPSUjfl0qKSIsCisgICAgICAgICAgICByIig/UDx1cmw+KGZpbGV8aHR0cHN8aHR0cHx3c3x3c3MpOi8vWy0wLTlhLXpBLVokXyshYCgpLC4/Lzs6Jj0lI35AXSopIiwKICAgICAgICAgKSwKICAgICBdCiAK", "rich_3006": "ZGlmZiAtLWdpdCBhL3JpY2gvcmVwci5weSBiL3JpY2gvcmVwci5weQppbmRleCA5ZGQ2NWNkZS4uOTUzMzEwMDYgMTAwNjQ0Ci0tLSBhL3JpY2gvcmVwci5weQorKysgYi9yaWNoL3JlcHIucHkKQEAgLTc2LDcgKzc2LDcgQEAgZGVmIGF1dG8oCiAgICAgICAgICAgICAgICAgICAgICAgICBwYXJhbS5QT1NJVElPTkFMX09SX0tFWVdPUkQsCiAgICAgICAgICAgICAgICAgICAgICAgICBwYXJhbS5LRVlXT1JEX09OTFksCiAgICAgICAgICAgICAgICAgICAgICk6Ci0gICAgICAgICAgICAgICAgICAgICAgICBpZiBwYXJhbS5kZWZhdWx0ID09IHBhcmFtLmVtcHR5OgorICAgICAgICAgICAgICAgICAgICAgICAgaWYgcGFyYW0uZGVmYXVsdCBpcyBwYXJhbS5lbXB0eToKICAgICAgICAgICAgICAgICAgICAgICAgICAgICB5aWVsZCBnZXRhdHRyKHNlbGYsIHBhcmFtLm5hbWUpCiAgICAgICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHlpZWxkIHBhcmFtLm5hbWUsIGdldGF0dHIoc2VsZiwgcGFyYW0ubmFtZSksIHBhcmFtLmRlZmF1bHQK"}""")
PD = Path('/kaggle/working/patches'); PD.mkdir(parents=True, exist_ok=True)
for tid, b64 in PATCHES.items():
    (PD / f'{tid}.patch').write_bytes(base64.b64decode(b64))
print('patches written:', len(PATCHES))

RUNNER = Path('/kaggle/working/replay_verify2.py')
RUNNER.write_bytes(base64.b64decode("""IiIiRmFpdGhmdWwgcmVwbGF5OiBtaXJyb3JzIHN3ZWdlbW1hJ3Mgb3duIHZlcmlmaWNhdGlvbiBydWxlLgoKc3dlZ2VtbWEgZGVyaXZlcyB0aGUgcmVxdWlyZWQgdGVzdCBub2RlcyBmcm9tIHRoZSB0ZXN0X3BhdGNoCihgX2V4dHJhY3RfdGVzdF9mdW5jdGlvbnNfZnJvbV9wYXRjaGApIGFuZCBydW5zIG9ubHkgdGhvc2UsIGZhaWxpbmcgb24KZmFpbGVkL2Vycm9yL3NraXBwZWQuIFRoaXMgc2NyaXB0IGRvZXMgZXhhY3RseSB0aGUgc2FtZSwgd2l0aCBubyBtb2RlbCBpbnZvbHZlZCwKc28gdGhlIHNhbWUgcGF0Y2hlcyBjYW4gYmUgcmVwbGF5ZWQgaW4gYW55IGVudmlyb25tZW50LgoiIiIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc2h1dGlsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRlbXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKVEFTS1MgPSBQYXRoKHN5cy5hcmd2WzFdKQpTTkFQUyA9IFBhdGgoc3lzLmFyZ3ZbMl0pClBBVENIRElSID0gUGF0aChzeXMuYXJndlszXSkKSURTID0gc3lzLmFyZ3ZbNDpdCk9VVCA9IFBhdGgob3MuZW52aXJvbi5nZXQoJ1JFUExBWV9PVVQnLCAnL291dC9yZXBsYXlfcmVzdWx0Lmpzb24nKSkKCnRhc2tzID0ge2pzb24ubG9hZHMobClbJ2luc3RhbmNlX2lkJ106IGpzb24ubG9hZHMobCkKICAgICAgICAgZm9yIGwgaW4gVEFTS1MucmVhZF90ZXh0KGVuY29kaW5nPSd1dGYtOCcpLnNwbGl0bGluZXMoKSBpZiBsLnN0cmlwKCl9ClRFU1RGSUxFID0gcmUuY29tcGlsZShyJ15cK1wrXCsgYi8oLio/KSQnLCByZS5NKQoKCmRlZiBleHRyYWN0X3Rlc3RzKHRlc3RfcGF0Y2g6IHN0cik6CiAgICAiIiJCeXRlLWlkZW50aWNhbCBsb2dpYyB0byBzd2VnZW1tYS5oYXJuZXNzLnZlcmlmaWNhdGlvbi5fZXh0cmFjdF90ZXN0X2Z1bmN0aW9uc19mcm9tX3BhdGNoLiIiIgogICAgaWYgbm90IHRlc3RfcGF0Y2g6CiAgICAgICAgcmV0dXJuIFtdCiAgICBhZGRlZCA9IHJlLmZpbmRhbGwocideXCtccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkKICAgIGlmIGFkZGVkOgogICAgICAgIHJldHVybiBsaXN0KGRpY3QuZnJvbWtleXMoYWRkZWQpKQogICAgcmVtb3ZlZCA9IHNldChyZS5maW5kYWxsKHInXi1ccyooPzphc3luY1xzKyk/ZGVmXHMrKHRlc3RfW0EtWmEtejAtOV9dKylccypcKCcsIHRlc3RfcGF0Y2gsIHJlLk1VTFRJTElORSkpCiAgICBodW5rX2hlYWRlcnMgPSByZS5maW5kYWxsKHInXkBAW15AXStAQFxzKig/OmFzeW5jXHMrKT9kZWZccysodGVzdF9bQS1aYS16MC05X10rKVxiJywgdGVzdF9wYXRjaCwgcmUuTVVMVElMSU5FKQogICAgY29udGV4dF9kZWZzID0gcmUuZmluZGFsbChyJ14gXHMqKD86YXN5bmNccyspP2RlZlxzKyh0ZXN0X1tBLVphLXowLTlfXSspXHMqXCgnLCB0ZXN0X3BhdGNoLCByZS5NVUxUSUxJTkUpCiAgICBjb250ZXh0X3NldCA9IHNldChjb250ZXh0X2RlZnMpCiAgICBjb21iaW5lZCA9IGxpc3QoZGljdC5mcm9ta2V5cyhodW5rX2hlYWRlcnMgKyBjb250ZXh0X2RlZnMpKQogICAgcmV0dXJuIFtmbiBmb3IgZm4gaW4gY29tYmluZWQgaWYgZm4gbm90IGluIHJlbW92ZWQgYW5kIG5vdCAoCiAgICAgICAgZm4gbm90IGluIGNvbnRleHRfc2V0IGFuZCAoYW55KHIuc3RhcnRzd2l0aChmbikgZm9yIHIgaW4gcmVtb3ZlZCkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvciBhbnkoYyAhPSBmbiBhbmQgYy5zdGFydHN3aXRoKGZuKSBmb3IgYyBpbiBjb250ZXh0X3NldCkpKV0KCgpkZWYgc2goY21kLCBjd2QpOgogICAgcmV0dXJuIHN1YnByb2Nlc3MucnVuKGNtZCwgY3dkPXN0cihjd2QpLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZW5jb2Rpbmc9J3V0Zi04JywgZXJyb3JzPSdyZXBsYWNlJywgdGltZW91dD0xODAwKQoKCnJlc3VsdHMgPSBbXQpmb3IgdGlkIGluIElEUzoKICAgIHJlYyA9IGRpY3QodGFzaz10aWQpCiAgICBzbmFwLCBwYXRjaCA9IFNOQVBTIC8gZid7dGlkfS50Z3onLCBQQVRDSERJUiAvIGYne3RpZH0ucGF0Y2gnCiAgICB0ID0gdGFza3MuZ2V0KHRpZCwge30pCiAgICB0cF90ZXh0ID0gdC5nZXQoJ3Rlc3RfcGF0Y2gnKSBvciAnJwogICAgaWYgbm90IHNuYXAuZXhpc3RzKCk6CiAgICAgICAgcmVjWydlcnJvciddID0gJ25vIHNuYXBzaG90JwogICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICBjb250aW51ZQogICAgd29yayA9IFBhdGgodGVtcGZpbGUubWtkdGVtcChwcmVmaXg9ZidyX3t0aWR9XycpKQogICAgdHJ5OgogICAgICAgIGlmIHNoKFsndGFyJywgJ3h6ZicsIHN0cihzbmFwKSwgJy1DJywgc3RyKHdvcmspXSwgd29yaykucmV0dXJuY29kZSAhPSAwOgogICAgICAgICAgICByZWNbJ2Vycm9yJ10gPSAnZXh0cmFjdCBmYWlsZWQnCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBwYXRjaC5leGlzdHMoKToKICAgICAgICAgICAgcCA9IHNoKFsnZ2l0JywgJ2FwcGx5JywgJy0td2hpdGVzcGFjZT1ub3dhcm4nLCBzdHIocGF0Y2gucmVzb2x2ZSgpKV0sIHdvcmspCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfcmMnXSA9IHAucmV0dXJuY29kZQogICAgICAgICAgICBpZiBwLnJldHVybmNvZGUgIT0gMDoKICAgICAgICAgICAgICAgIHAgPSBzaChbJ2dpdCcsICdhcHBseScsICctLTN3YXknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cihwYXRjaC5yZXNvbHZlKCkpXSwgd29yaykKICAgICAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfM3dheV9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHJlY1snYWdlbnRfcGF0Y2hfdGFpbCddID0gKChwLnN0ZGVyciBvciAnJykgKyAocC5zdGRvdXQgb3IgJycpKVstMjAwOl0KICAgICAgICB0cCA9IHdvcmsucGFyZW50IC8gZid7dGlkfV90ZXN0LnBhdGNoJwogICAgICAgIHRwLndyaXRlX2J5dGVzKCh0cF90ZXh0IGlmIHRwX3RleHQuZW5kc3dpdGgoJ1xuJykgZWxzZSB0cF90ZXh0ICsgJ1xuJykKICAgICAgICAgICAgICAgICAgICAgICAucmVwbGFjZSgnXHJcbicsICdcbicpLmVuY29kZSgndXRmLTgnKSkKICAgICAgICBwID0gc2goWydnaXQnLCAnYXBwbHknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cih0cCldLCB3b3JrKQogICAgICAgIHJlY1sndGVzdF9wYXRjaF9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgaWYgcC5yZXR1cm5jb2RlICE9IDA6CiAgICAgICAgICAgIHAgPSBzaChbJ2dpdCcsICdhcHBseScsICctLTN3YXknLCAnLS13aGl0ZXNwYWNlPW5vd2FybicsIHN0cih0cCldLCB3b3JrKQogICAgICAgICAgICByZWNbJ3Rlc3RfcGF0Y2hfM3dheV9yYyddID0gcC5yZXR1cm5jb2RlCiAgICAgICAgICAgIHJlY1sndGVzdF9wYXRjaF90YWlsJ10gPSAoKHAuc3RkZXJyIG9yICcnKSArIChwLnN0ZG91dCBvciAnJykpWy0yMDA6XQoKICAgICAgICBmaWxlcyA9IHNvcnRlZCh7Zi5zdHJpcCgpIGZvciBmIGluIFRFU1RGSUxFLmZpbmRhbGwodHBfdGV4dCl9KQogICAgICAgIGZucyA9IGV4dHJhY3RfdGVzdHModHBfdGV4dCkKICAgICAgICByZWNbJ3Rlc3RfZmlsZXMnXSA9IGZpbGVzCiAgICAgICAgcmVjWyd0ZXN0X2ZucyddID0gZm5zCiAgICAgICAgbm9kZXMgPSBbZid7Zn06Ontmbn0nIGZvciBmIGluIGZpbGVzIGZvciBmbiBpbiBmbnNdIG9yIGZpbGVzCiAgICAgICAgaWYgbm90IG5vZGVzOgogICAgICAgICAgICByZWNbJ2Vycm9yJ10gPSAnbm8gbm9kZXMnCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwID0gc2goW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncHl0ZXN0JywgJy1xJywgJy0tbm8taGVhZGVyJywgJy1wJywgJ25vOmNhY2hlcHJvdmlkZXInXSArIG5vZGVzLCB3b3JrKQogICAgICAgIHRhaWwgPSAocC5zdGRvdXQgb3IgJycpICsgKHAuc3RkZXJyIG9yICcnKQogICAgICAgIG0gPSByZS5maW5kYWxsKHInKFxkKylccysocGFzc2VkfGZhaWxlZHxlcnJvcnxlcnJvcnN8c2tpcHBlZCknLCB0YWlsKQogICAgICAgIHJlY1snY291bnRzJ10gPSB7azogaW50KHYpIGZvciB2LCBrIGluIG19CiAgICAgICAgcmVjWydub2RlcyddID0gbm9kZXMKICAgICAgICByZWNbJ3B5dGVzdF90YWlsJ10gPSB0YWlsWy0zMDA6XQogICAgICAgIHBhc3NlZCA9IHJlY1snY291bnRzJ10uZ2V0KCdwYXNzZWQnLCAwKQogICAgICAgIGJhZCA9IHN1bShyZWNbJ2NvdW50cyddLmdldChrLCAwKSBmb3IgayBpbiAoJ2ZhaWxlZCcsICdlcnJvcicsICdlcnJvcnMnLCAnc2tpcHBlZCcpKQogICAgICAgIHJlY1sncmVzb2x2ZWRfaGVyZSddID0gYm9vbChwYXNzZWQgPiAwIGFuZCBiYWQgPT0gMCkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICByZWNbJ2Vycm9yJ10gPSBmJ3t0eXBlKGUpLl9fbmFtZV9ffToge2V9JwogICAgZmluYWxseToKICAgICAgICBzaHV0aWwucm10cmVlKHdvcmssIGlnbm9yZV9lcnJvcnM9VHJ1ZSkKICAgIHJlc3VsdHMuYXBwZW5kKHJlYykKCk9VVC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQpPVVQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdHMsIGluZGVudD0xKSwgZW5jb2Rpbmc9J3V0Zi04JykKcHJpbnQoJyUtMjJzICUtN3MgJS03cyAlLTlzICVzJyAlICgndGFzaycsICdhX3JjJywgJ3RfcmMnLCAncmVzb2x2ZWQnLCAnY291bnRzIC8gZXJyJykpCmZvciByIGluIHJlc3VsdHM6CiAgICBwcmludCgnJS0yMnMgJS03cyAlLTdzICUtOXMgJXMnICUgKHIuZ2V0KCd0YXNrJyksIHIuZ2V0KCdhZ2VudF9wYXRjaF9yYycpLCByLmdldCgndGVzdF9wYXRjaF9yYycpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByLmdldCgncmVzb2x2ZWRfaGVyZScpLCByLmdldCgnY291bnRzJykgb3IgKHIuZ2V0KCdlcnJvcicpIG9yICcnKVs6NDVdKSkKb2sgPSBzdW0oMSBmb3IgciBpbiByZXN1bHRzIGlmIHIuZ2V0KCdyZXNvbHZlZF9oZXJlJykpCnByaW50KCdcbnJlc29sdmVkIGhlcmU6ICVkIC8gJWQnICUgKG9rLCBsZW4ocmVzdWx0cykpKQpwcmludCgnd3JvdGUnLCBPVVQpCg=="""))

env = dict(os.environ, REPLAY_OUT='/kaggle/working/replay_result.json')
p = subprocess.run([sys.executable, str(RUNNER), str(TASKS), str(SNAPS), str(PD)] + sorted(PATCHES),
                   capture_output=True, text=True, env=env)
print(p.stdout)
print('STDERR', p.stderr[-1500:])
res = json.loads(Path('/kaggle/working/replay_result.json').read_text())
print('=== KAGGLE-SIDE REPLAY ===')
for r in res:
    print('%-22s resolved=%-6s counts=%s err=%s' % (r['task'], r.get('resolved_here'),
          r.get('counts'), (r.get('error') or '')[:40]))
print('resolved on Kaggle: %d / %d' % (sum(1 for r in res if r.get('resolved_here')), len(res)))
